
## Build Races Dimension
1. Read silver races table
2. Read silver circuits table
3. Join the data from races with circuits using circuit_id
4. Select the required columns
-     races.season
-     races.round
-     races.race_name
-     races.race_date
-     circuits.circuit_name
-     circuits.locality
-     circuits.country
5. Write the transformed data to gold dim_races table



Below changes are required to implement Incremental Load Processing

1. Accept batch_id as a parameter to the notebook
2. Process data for only the batch_id being passed in (i.e., filter reading from silver using the batch_id)
Add created_timestamp, updated_timestamp to the gold table.
3. Merge the processed data to the gold table
4. created_timestamp should only be populated at the time of inserting/ creating the record. It should not be updated during the merge update.

In [0]:
# 1. Get batch ID
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/04_gold_helpers"

In [0]:
from pyspark.sql import functions as f

In [0]:
gold_path = "s3://bizf1project123/Silver/Kalashri_increment/dim_races"

## Step 1 - Read source tables
- circuits
- races

In [0]:
races_path = f"{s3_root_path}/Silver/Kalashri_increment/races"

circuits_path = f"{s3_root_path}/Silver/Kalashri_increment/circuits"

target_path = f"{s3_root_path}/Gold/Kalashri_increment/dim_races"

In [0]:
races_df = spark.read.format("delta").load(races_path)
circuits_df = spark.read.format("delta").load(circuits_path)

display(races_df.limit(20))
display(circuits_df.limit(20))

## Step 2 - Join races with circuits using circuit_id
Select the following columns

1. races.season
2. races.round
3. races.race_name
4. races.race_date
5. circuits.circuit_name
6. circuits.locality
7. circuits.country

In [0]:
dim_races_df = (
            races_df
                .join(
                    circuits_df,
                    races_df.circuit_id == circuits_df.circuit_id,
                    "inner"
                )
                .select (
                    races_df.season,
                    races_df.round,
                    races_df.race_name,
                    races_df.race_date,
                    circuits_df.circuit_name,
                    circuits_df.locality,
                    circuits_df.country
                )
        )

display(dim_races_df)

## Step 3 - Write the transformed data to the gold dim_races table

In [0]:
write_to_gold(
    input_df=dim_races_df,
    target_path=target_path,
    merge_condition="t.season = s.season AND t.round = s.round",
    columns_to_update=[
        "race_name",
        "race_date",
        "circuit_name",
        "locality",
        "country"
    ]
)

In [0]:
%sql
SELECT *
FROM delta.`s3://bizf1project123/Gold/Kalashri_increment/dim_races`;

In [0]:
# dbutils.fs.rm(
#     "s3://bizf1project123/Gold/Kalashri_increment/dim_races",
#     True
# )